# Text Representation Techniques

In [1]:
!pip install nltk
!pip install pandas
!pip install numpy
!pip install scipy
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import sys
import string
import nltk
import sklearn
import numpy as np
from typing import Iterable

In [3]:
print(f'sklearn.__version__={sklearn.__version__}')

sklearn.__version__=1.7.2


## Stemming

Stemming consists on removing the suffixes or prefixes used in word. The returned string from a stemmer might not be a valid word from the language.

In [6]:
# spacy
from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer

porter    = PorterStemmer()
lancaster = LancasterStemmer()
nltk.download("wordnet")
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [8]:
from nltk.tokenize import sent_tokenize, word_tokenize

def stem(sentence):
    token_words = word_tokenize(sentence)
    sentence_stemmed = []
    for word in token_words:
        sentence_stemmed.append(porter.stem(word))
        sentence_stemmed.append(" ")
    return "".join(sentence_stemmed)

In [9]:
s = "J.K. Rowling wrote Harry Potter. She never expected the book to be famous."
stem(s)

'j.k. rowl wrote harri potter . she never expect the book to be famou . '

In [ ]:
sent_tokenize(s)

In [ ]:
# Be carefull separating phrases
s.split(".")

In [ ]:
word_tokenize(s)

## Lemmatization

Lemmatization consists on properly use of a vocabulary and morphological analysis of words, aiming to remove inflectional endings only with the goal of returning any word to a set of base (or dictionary form) words.

`Lemmatize(saw) = see`

We will use a lemmatizer from WordNet (https://wordnet.princeton.edu) avaliable from nltk.

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
wordnet_lemmatizer = WordNetLemmatizer()

In [ ]:
sentence = "I was running and eating. This was a terrible idea."
punctuations="?:!.,;"
sentence_words = nltk.word_tokenize(sentence)
for word in sentence_words:
    if word in punctuations:
        sentence_words.remove(word)


In [ ]:
sentence_words
print("{0:20}{1:20}".format("Word","Lemma"))
for word in sentence_words:
    print ("{0:20}{1:20}".format(word,wordnet_lemmatizer.lemmatize(word)))

Notice that the words did not change!

This is because there was no context. If we give a part of speech type then the lemmatizer will do what we expect (if no POS is provided, "noun" is assumed).

The WordNet lemmatizer requires POS because the correct base form (lemma) of a word depends on what part of speech it is. The same word can have different lemmas depending on its role.

For example, the word "better":
- As an adjective: lemma is "good"
- As an adverb: lemma is "well"
- As a verb: lemma is "better" (to better something)

In [ ]:
print(wordnet_lemmatizer.lemmatize("running"))           # "running" (wrong — treated as noun)
print(wordnet_lemmatizer.lemmatize("running", pos="v"))  # "run" (correct)

# Building a Simple Document Classifier

Let us build a simple document classifier featurizing each document by word counts

## Load Data

In [ ]:
import sklearn.linear_model
import sklearn.model_selection
import sklearn.pipeline
import sklearn.feature_extraction
import sklearn.datasets
import scipy
import scipy.sparse as sp
import numpy as np
from typing import Iterable
import sklearn
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
X = sklearn.datasets.fetch_20newsgroups()

X_train = sklearn.datasets.fetch_20newsgroups(subset="train").data
y_train = sklearn.datasets.fetch_20newsgroups(subset="train").target
X_test  = sklearn.datasets.fetch_20newsgroups(subset="test").data
y_test  = sklearn.datasets.fetch_20newsgroups(subset="test").target

In [ ]:
X_train = X_train[:5000]
y_train = y_train[:5000]
print(len(X_train), len(y_train))
print(type(X_train), type(y_train))

In [ ]:
x = X_train[0]

In [ ]:
np.unique(y_train)

In [ ]:
print(x)

In [ ]:
y_train[0]

## Tiny function to create a feature matrix of word counts (feature counting)


In [ ]:
from collections import defaultdict
docs = [['hello', 'world', 'hello'], ['goodbye', 'cruel', 'teacher', 'goodbye']]

def prepare_word_counts_with_dict(docs: Iterable[str], verbose=False):
    ind_ptr = [0]
    ind_col = []
    data = []
    vocabulary = {}
    
    for m, doc in enumerate(docs):
        word_ind_counter = defaultdict(int)  # document counter for each doc in X
        for word in doc: 
            vocabulary.setdefault(word, len(vocabulary))
            word_ind_counter[word] += 1
                
        data.extend(word_ind_counter.values())
        ind_ptr.append(ind_ptr[-1] + len(word_ind_counter))
        ind_col.extend([vocabulary[w] for w in word_ind_counter.keys()])

    if verbose:
        print('len vocab =', len(vocabulary))
        print('vocab =', vocabulary)
        print('data =', data)
        print('ind_ptr =', ind_ptr)
        print('ind_col =', ind_col)
        
    return (data, ind_col, ind_ptr)

In [ ]:
prepare_word_counts_with_dict(docs)

In [ ]:
sp.csr_matrix(prepare_word_counts_with_dict(docs)).toarray()

We can create a bigger dataset to benchmark

In [ ]:
docs_big = docs * 1000

In [ ]:
%%timeit 
prepare_word_counts_with_dict(docs_big)

In [ ]:
sp.csr_matrix(prepare_word_counts_with_dict(docs_big))

## Customizing Vectorizer Classes

- **preprocessor**: a callable that takes an entire document as input (as a single string), and returns a possibly transformed version of the document, still as an entire string. This can be used to remove HTML tags, lowercase the entire document, etc.


- **tokenizer**: a callable that takes the output from the preprocessor and splits it into tokens, then returns a list of these.


- **analyzer**: a callable that replaces the preprocessor and tokenizer. The default analyzers all call the preprocessor and tokenizer, but custom analyzers will skip this. N-gram extraction and stop word filtering take place at the analyzer level, so a custom analyzer may have to reproduce these steps.

Notice that in order to build our data as a matrix we need to use sparse matrices due to the high dimensionality (number of words/features) of the vocabulary.

### No Stemmer and no doc_cleaner

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vanilla_count_vectorizer = CountVectorizer()

logistic = sklearn.linear_model.LogisticRegression(C=0.1)

model_pipe_0 = sklearn.pipeline.Pipeline([("countvectorizer", vanilla_count_vectorizer),
                                         ("logisticregression", logistic)])

In [ ]:
model_pipe_0.steps

In [ ]:
model_pipe_0.fit(X_train[0:100],y_train[0:100])

In [ ]:
model_pipe_0.predict(X_train[0:100])

In [ ]:
%%time
model_pipe_0.fit(X_train,y_train)
y_test_pred  = model_pipe_0.predict(X_test)
y_train_pred = model_pipe_0.predict(X_train)

In [ ]:
model_pipe_0.steps[0][1].transform(X_train)

In [ ]:
acc_train_0 = np.mean(y_train == y_train_pred)
acc_test_0 = np.mean(y_test == y_test_pred)
print("Accuracy train: {}    Accuracy test: {}".format(acc_train_0, acc_test_0))

### No stemmer but doc_cleaner

In [ ]:
import re

def make_cleaner(pattern):
    def clean(text):
        return re.sub(pattern, '', text).lower()
    return clean

simple_count_vectorizer = CountVectorizer(preprocessor=make_cleaner(r'[^a-zA-Z\s]'))
logistic = sklearn.linear_model.LogisticRegression(C=0.1)

model_pipe_1 = sklearn.pipeline.Pipeline([("countvectorizer", simple_count_vectorizer),
                                        ("logisticregression", logistic)])

In [ ]:
%%time
model_pipe_1.fit(X_train,y_train)

In [ ]:
y_test_pred  = model_pipe_1.predict(X_test)
y_train_pred = model_pipe_1.predict(X_train)

acc_train_1 = np.mean(y_train == y_train_pred)
acc_test_1 = np.mean(y_test == y_test_pred)

print("Accuracy train: {}    Accuracy test: {}".format(acc_train_1, acc_test_1))

### Use a SnowballStemmer

In [ ]:
from nltk.stem import SnowballStemmer

stemmer = SnowballStemmer("english")

def make_cleaner(pattern):
    def clean(text):
        return re.sub(pattern, '', text).lower()
    return clean

def stemming_tokenizer(text):
    return [stemmer.stem(word) for word in text.split()]

simple_count_vectorizer_stemmer = CountVectorizer(
    preprocessor=make_cleaner(r'[^a-zA-Z\s]'),
    tokenizer=stemming_tokenizer
)

logistic = sklearn.linear_model.LogisticRegression(C=0.1)

model_pipe_2 = sklearn.pipeline.Pipeline([("countvectorizer", simple_count_vectorizer_stemmer),
                                        ("logisticregression", logistic)],
                                         )

In [ ]:
%%time
model_pipe_2.fit(X_train,y_train)

In [ ]:
y_test_pred  = model_pipe_2.predict(X_test)
y_train_pred = model_pipe_2.predict(X_train)

acc_train_2 = np.mean(y_train == y_train_pred)
acc_test_2  = np.mean(y_test == y_test_pred)

print("Accuracy train: {}    Accuracy test: {}".format(acc_train_2, acc_test_2))

###   Ngram features with Sklearn vectorizer

In [ ]:
count_vectorizer = sklearn.feature_extraction.text.CountVectorizer(ngram_range=(1,2))
logistic = sklearn.linear_model.LogisticRegression(C=0.1)

model_pipe_3 = sklearn.pipeline.Pipeline([("countvectorizer", count_vectorizer),
                                          ("logisticregression", logistic)])

In [ ]:
%%time
model_pipe_3.fit(X_train,y_train)

In [ ]:
y_test_pred  = model_pipe_3.predict(X_test)
y_train_pred = model_pipe_3.predict(X_train)

acc_train_3 = np.mean(y_train == y_train_pred)
acc_test_3  = np.mean(y_test == y_test_pred)

print("Accuracy train: {}    Accuracy test: {}".format(acc_train_4, acc_test_4))

In [ ]:
model_pipe_3.steps[0][1].transform(X_train[0:1])

In [ ]:
df_results["sklearn countvectorizer 2gram"] = [acc_train_3, acc_test_3]

In [ ]:
df_results

In [ ]:
%matplotlib inline
#df_results.T["test"].plot(kind="barh", xlim=(0.79,0.83))
df_results.T["test"].plot(kind="barh")

### Table with results for each pipeline

In [ ]:
import pandas as pd

In [ ]:
df_results = pd.DataFrame()
df_results["no clean no stem"]   = [acc_train_0, acc_test_0]
df_results["yes clean no stem"]  = [acc_train_1, acc_test_1]
df_results["yes clean yes stem"] = [acc_train_2, acc_test_2]
df_results["ngrams"] = [acc_train_3, acc_test_3]
df_results.index=["train","test"] 

In [ ]:
df_results

##  Feature selection: SelectKbest

In [ ]:
from sklearn.feature_selection import SelectKBest

count_vectorizer = sklearn.feature_extraction.text.CountVectorizer(ngram_range=(1,2))
#feature_selector = SelectKBest(chi2, k = 700000)
feature_selector = SelectKBest(chi2, k = 400000)
logistic = sklearn.linear_model.LogisticRegression(C=0.1)

model_pipe_4 = sklearn.pipeline.Pipeline([("count_vectorizer", count_vectorizer),
                                          ("feature_selector", feature_selector),
                                          ("logisticregression", logistic)])

In [ ]:
%%time
model_pipe_4.fit(X_train, y_train)

In [ ]:
acc_train = np.mean(model_pipe_4.predict(X_train) == y_train)
acc_test = np.mean(model_pipe_4.predict(X_test) == y_test)
df_results["sklearn countvectorizer 2gram + selection"] = [acc_train, acc_test]

In [ ]:
#df_results.T["test"].plot(kind="barh", xlim=(0.79,0.83))
df_results.T["test"].plot(kind="barh", xlim=(0.72,0.75))